# DecisionMaster Demo

This notebook loads the published Hugging Face model and demonstrates the inference API:

1. single decision: `query` + `candidates` (+ optional `context`)
2. ranking and probabilities for a decision
3. batched prediction and throughput

In [1]:
# %pip install -e ..
from decision_master import DecisionMaster, DEFAULT_MODEL_ID

# Loads leobitz/decision-master @ base. A local checkpoint dir (e.g. "../checkpoints/best_val") also works.
model = DecisionMaster.from_pretrained(DEFAULT_MODEL_ID)
model.device, next(model.model.qwen.parameters()).dtype

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.pt:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

The tokenizer you are loading from '/home/leo/.cache/huggingface/hub/models--leobitz--decision-master-base/snapshots/785bb181af7974b52cf9cac68ae1cebf15ec095e' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(device(type='cuda'), torch.bfloat16)

In [2]:
context = "Customer says they were billed twice for the same order and wants a refund."
query = "What is the best category?"

for choices in (
    ["billing", "shipping", "technical", "other"],
    ["billing", "shipping", "technical", "refund"],
):
    pred = model.decide(query=query, candidates=choices, context=context)
    print(f"{pred.best!r} with probability {pred.probabilities[pred.best_index]:.3f}  <- {choices}")

'other' with probability 0.990  <- ['billing', 'shipping', 'technical', 'other']
'refund' with probability 0.967  <- ['billing', 'shipping', 'technical', 'refund']


In [3]:
pred = model.decide(
    query="Which team should take ownership of this case?",
    context="A customer was charged twice, emailed support twice without a reply, and wants the duplicate charge refunded.",
    candidates=["billing", "support", "technical", "refunding"],
)
for label, p in pred.ranked():
    print(f"{label:12s} {p:.3f}")

support      0.511
billing      0.330
refunding    0.089
technical    0.071


In [4]:
# No context: only the query is used.
pred = model.decide(query="Which of these is a fruit?", candidates=["apple", "table", "car"])
print(pred.best, pred.probabilities)

apple [0.5248638987541199, 0.06862631440162659, 0.4065098166465759]


In [5]:
# Many decisions with different candidate counts; results come back in input order.
decisions = [
    {"query": "Is the sentiment positive or negative?", "context": "The food was cold and the service was slow.",
     "candidates": ["positive", "negative"]},
    {"query": "What is the capital of France?", "candidates": ["Berlin", "Paris", "Madrid", "Rome", "Lisbon"]},
    {"query": "How urgent is this case?", "context": "The production database is down for all customers.",
     "candidates": ["Low", "Medium", "High"]},
]
for d, pred in zip(decisions, model.predict(decisions)):
    print(f"{d['query']}  ->  {pred.best}  ({max(pred.probabilities):.3f})")

Is the sentiment positive or negative?  ->  negative  (0.999)
What is the capital of France?  ->  Paris  (0.721)
How urgent is this case?  ->  High  (0.383)


In [6]:
import time
import torch

rows = [
    {"context": "word " * 150, "query": f"question {i}", "candidates": [f"answer option number {j}" for j in range(4)]}
    for i in range(512)
]
model.predict(rows[:32])  # warm-up
if model.device.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
model.predict(rows, batch_size=32)
if model.device.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start
print(f"{len(rows) / elapsed:.1f} decisions/s")

149.4 decisions/s
